# LiciGuard · Bajar bases REALES del SEACE para probar

Este cuaderno consigue **bases de licitaciones reales** para probar LiciGuard y medir qué tan bien acierta.

**¿De dónde salen?** Cada proceso de `data/ocds/historial.csv` (datos abiertos del OECE) trae el enlace al
primer documento publicado en el SEACE, que normalmente son las **bases** o **bases integradas**.
El cuaderno elige una muestra variada, las descarga con calma (una cada pocos segundos, para no saturar el SEACE)
y se queda solo con las que sirven: **PDF con texto** que parezcan bases.

| Paso | Qué hace | Resultado |
|---|---|---|
| 0 | Prepara Colab (sube `liciguard-ai.zip`) | Proyecto listo |
| 1 | Eliges rubro, desde qué año y cuántas | Lista de procesos |
| 2 | Descarga y revisa cada documento | PDFs en `data/bases/` + `descargas.csv` |
| 3 | Prueba LiciGuard en todas | Tabla con la recomendación de cada una |
| 4 | Descarga los resultados | ZIP con las bases útiles |

Después, en el cuaderno **03 (Parte B y D)**, marcas si cada dato encontrado es correcto y obtienes el % de aciertos.

> **Importante:** son documentos públicos. Úsalos solo para pruebas; no los publiques como tuyos.
> Algunos enlaces pueden no descargarse (el SEACE a veces pide navegador, o el archivo es RAR/Word o escaneado):
> el cuaderno lo indica y sigue con el siguiente.

## Paso 0 · Solo en Google Colab: sube `liciguard-ai.zip` (el proyecto completo, trae `historial.csv`)

In [ ]:
import os, sys, subprocess, zipfile
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import files
    if not os.path.exists("/content/liciguard-ai/core"):
        print("Sube el archivo liciguard-ai.zip 👇")
        for nombre in files.upload():
            if nombre.endswith(".zip"):
                zipfile.ZipFile(nombre).extractall("/content")
    os.chdir("/content/liciguard-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pymupdf", "pandas"], check=True)
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
import pandas as pd
from core.bases_reales import elegir_procesos, descargar, procesar_descarga, revisar_pdf
print("Proyecto en:", RAIZ)

## Paso 1 · Elige qué bases quieres

- `RUBRO`: uno de los rubros de tu historial (la celda te muestra cuáles hay).
- `DESDE`: año desde el que buscar (las más recientes usan la Ley 32069).
- `CUANTAS`: cuántas intentar. Empieza con **15**; no todas servirán (algunas son RAR, Word o escaneadas).
- `MAX_POR_ENTIDAD`: para que haya variedad de entidades.

In [ ]:
RUBRO = "Uniformes y textiles"
DESDE = "2025"
CUANTAS = 15
MAX_POR_ENTIDAD = 2

hist = pd.read_csv(RAIZ / "data" / "ocds" / "historial.csv", dtype=str, keep_default_na=False)
print("Rubros disponibles:", sorted(hist["rubro"].unique()))
muestra = elegir_procesos(hist, RUBRO, DESDE, CUANTAS, MAX_POR_ENTIDAD)
print(f"Procesos elegidos: {len(muestra)}")
muestra[["fecha", "entidad", "titulo", "valor_estimado"]]

## Paso 2 · Descargar y revisar (tarda unos segundos por archivo: deja la pestaña abierta)

Se guarda en `data/bases/` solo lo que **sirve**. Lo demás queda anotado en `data/bases/descargas.csv` con el motivo.

In [ ]:
import re, time
carpeta = RAIZ / "data" / "bases"
carpeta.mkdir(parents=True, exist_ok=True)
PAUSA_ENTRE_DESCARGAS = 3   # segundos: sé amable con el SEACE

registro = []
for i, p in muestra.iterrows():
    fila = {"ocid": p["ocid"], "fecha": p["fecha"], "entidad": p["entidad"], "titulo": p["titulo"][:120],
            "url": p["url"], "archivo": "", "tipo": "", "paginas": 0, "con_texto": 0, "motivo": ""}
    try:
        tipo, pdfs = procesar_descarga(descargar(p["url"]))
        fila["tipo"] = tipo
        if not pdfs:
            fila["motivo"] = {"html": "el SEACE pidió navegador (no se pudo bajar directo)",
                              "rar": "viene en RAR (descomprímelo a mano)",
                              "doc": "es un Word, no un PDF"}.get(tipo, f"formato {tipo}")
        for nombre, data in pdfs[:1]:   # el primer PDF (los que dicen "bases" van primero)
            r = revisar_pdf(data)
            fila.update(paginas=r["paginas"], con_texto=r["con_texto"], motivo=r["motivo"])
            if r["motivo"] == "sirve":
                limpio = re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{p['fecha']}_{p['entidad'][:30]}_{nombre}")[:110]
                (carpeta / limpio).write_bytes(data)
                fila["archivo"] = limpio
    except Exception as e:
        fila["motivo"] = f"no se pudo descargar: {type(e).__name__}"
    registro.append(fila)
    print(f"{i + 1:>2}/{len(muestra)} · {fila['motivo']:<45} · {p['entidad'][:40]}")
    time.sleep(PAUSA_ENTRE_DESCARGAS)

registro = pd.DataFrame(registro)
registro.to_csv(carpeta / "descargas.csv", index=False)
print(f"\n✅ Bases útiles: {(registro['motivo'] == 'sirve').sum()} de {len(registro)} (detalle en data/bases/descargas.csv)")
registro["motivo"].value_counts()

## Paso 3 · Probar LiciGuard en las bases descargadas

Usa un perfil de ejemplo. Verás la recomendación y lo que encontró en cada una.
Para **medir aciertos**, sigue luego con el cuaderno **03 → Parte B** (crea `revision.csv`) y **Parte D**.

In [ ]:
from core.buscador import Normativa
from core.historial_ocds import cargar_historial
from core.pdf_parser import extraer_paginas
from core.preguntas import analizar

PERFIL_DEMO = {"experiencia_soles": 320000, "certificaciones": [], "rnp_vigente": True, "es_mype": True}
norm, historial = Normativa.cargar(), cargar_historial()
filas = []
for pdf in sorted(carpeta.glob("*.pdf")):
    r = analizar(extraer_paginas(pdf.read_bytes()), PERFIL_DEMO, norm, historial)
    d = r["datos"]
    filas.append({"archivo": pdf.name[:60], "entidad": d.get("entidad"), "valor_estimado": d.get("valor_estimado"),
                  "recomendacion": r["recomendacion"]["codigo"], "alertas": len(r["justa"]["hallazgos"]),
                  "requisitos_encontrados": len(r["requisitos"]),
                  "cronograma": ", ".join(r["cronograma"]) or "—"})
resultados = pd.DataFrame(filas)
resultados.to_csv(carpeta / "resultados_liciguard.csv", index=False)
resultados

**Cómo leer la tabla:** si en muchas bases sale `valor_estimado` vacío o `cronograma` = —, esos son los patrones
que hay que mejorar en `core/extraccion.py` (ver *docs/REVISION_TECNICA.md*, sección 6.3). Anótalo: es justo lo que
le sirve al programador.

## Paso 4 · Descargar los resultados (Colab)

In [ ]:
import shutil
if EN_COLAB:
    shutil.make_archive("/content/bases_reales", "zip", carpeta)
    files.download("/content/bases_reales.zip")
else:
    print("En tu computadora los archivos ya quedaron en", carpeta)